## Step 1 — Dataset Exploration
**Dissertation: TinyML-Based Anomaly Detection**  
**Author: Thondupu Dileep | 2024AB05233**

This notebook explores both datasets before preprocessing:
- CWRU Bearing Dataset (vibration .mat files)
- MIMII Dataset (machine sound .wav files)

In [ ]:
import numpy as np
import pandas as pd
import scipy.io as sio
import librosa
import librosa.display
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)

---
## Part 1 — CWRU Bearing Dataset

In [ ]:
# list all mat files
CWRU_DIR = '../CWRU/raw'
files = sorted(os.listdir(CWRU_DIR))
print(f'Total files: {len(files)}')
for f in files:
    size = os.path.getsize(os.path.join(CWRU_DIR, f)) / 1e6
    print(f'  {f}  ({size:.1f} MB)')

In [ ]:
# load and inspect normal file
normal_file = os.path.join(CWRU_DIR, 'Time_Normal_1_098.mat')
mat = sio.loadmat(normal_file)

print('Keys in mat file:')
for k in mat.keys():
    if not k.startswith('__'):
        print(f'  {k}: shape={mat[k].shape}, dtype={mat[k].dtype}')

In [ ]:
def get_signal(filepath):
    mat = sio.loadmat(filepath)
    for k in mat.keys():
        if 'DE_time' in k:
            return mat[k].flatten().astype(np.float32)
    for k in mat.keys():
        if 'time' in k.lower() and not k.startswith('__'):
            return mat[k].flatten().astype(np.float32)
    return None

signal = get_signal(normal_file)
print(f'Signal length: {len(signal):,} samples')
print(f'Duration: {len(signal)/48000:.1f} seconds at 48 kHz')
print(f'Min: {signal.min():.4f}  Max: {signal.max():.4f}  Mean: {signal.mean():.4f}')

In [ ]:
# plot a short segment of the normal signal
SR = 48000
t  = np.arange(len(signal)) / SR

plt.figure(figsize=(12, 3))
plt.plot(t[:SR], signal[:SR], linewidth=0.5)  # first 1 second
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title('Normal Vibration Signal — First 1 Second')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# compare all fault types — first 0.1 second
file_map = {
    'Normal'        : 'Time_Normal_1_098.mat',
    'Ball 0.007"'   : 'B007_1_123.mat',
    'Ball 0.021"'   : 'B021_1_227.mat',
    'Inner Race 007': 'IR007_1_110.mat',
    'Inner Race 021': 'IR021_1_214.mat',
    'Outer Race 007': 'OR007_6_1_136.mat',
    'Outer Race 021': 'OR021_6_1_239.mat',
}

n_plot  = 4800   # 0.1 second
fig, axes = plt.subplots(len(file_map), 1, figsize=(12, 14), sharex=True)

for ax, (label, fname) in zip(axes, file_map.items()):
    fpath = os.path.join(CWRU_DIR, fname)
    if not os.path.exists(fpath):
        continue
    sig = get_signal(fpath)
    t2  = np.arange(n_plot) / SR
    ax.plot(t2, sig[:n_plot], linewidth=0.6)
    ax.set_ylabel(label, fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Vibration Signals — All Fault Types (0.1 second)', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# FFT frequency spectrum comparison — normal vs one fault
from scipy.fft import rfft, rfftfreq

n_seg = 4096
seg_normal = get_signal(os.path.join(CWRU_DIR, 'Time_Normal_1_098.mat'))[:n_seg]
seg_fault  = get_signal(os.path.join(CWRU_DIR, 'IR021_1_214.mat'))[:n_seg]

freqs = rfftfreq(n_seg, d=1/SR)
fft_n = np.abs(rfft(seg_normal)) / n_seg
fft_f = np.abs(rfft(seg_fault))  / n_seg

plt.figure(figsize=(12, 4))
plt.plot(freqs, fft_n, label='Normal',          alpha=0.8, linewidth=0.8)
plt.plot(freqs, fft_f, label='Inner Race 021"', alpha=0.8, linewidth=0.8)
plt.xlim(0, 10000)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude')
plt.title('FFT Spectrum — Normal vs Inner Race Fault')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# summary stats for all files
rows = []
for label, fname in file_map.items():
    fpath = os.path.join(CWRU_DIR, fname)
    if not os.path.exists(fpath):
        continue
    sig = get_signal(fpath)
    rms = np.sqrt(np.mean(sig**2))
    from scipy.stats import kurtosis
    rows.append({
        'Fault Type' : label,
        'Samples'    : len(sig),
        'RMS'        : round(float(rms), 5),
        'Kurtosis'   : round(float(kurtosis(sig)), 3),
        'Peak'       : round(float(np.max(np.abs(sig))), 5)
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
# how many windows will we get from each file?
WINDOW = 2048
STEP   = int(WINDOW * 0.5)  # 50% overlap
print(f'Window size: {WINDOW} samples ({WINDOW/SR*1000:.1f} ms)')
print(f'Step size  : {STEP} samples\n')
print(f'{"Fault":<20} {"Signal len":>12} {"Windows":>10}')
print('-' * 45)
for label, fname in file_map.items():
    fpath = os.path.join(CWRU_DIR, fname)
    if not os.path.exists(fpath):
        continue
    sig  = get_signal(fpath)
    wins = (len(sig) - WINDOW) // STEP + 1
    print(f'{label:<20} {len(sig):>12,} {wins:>10}')

---
## Part 2 — MIMII Dataset

In [ ]:
NORMAL_DIR   = '../MIMII/normal'
ABNORMAL_DIR = '../MIMII/abnormal'

n_normal   = len([f for f in os.listdir(NORMAL_DIR)   if f.endswith('.wav')])
n_abnormal = len([f for f in os.listdir(ABNORMAL_DIR) if f.endswith('.wav')])

print(f'Normal files  : {n_normal}')
print(f'Abnormal files: {n_abnormal}')
print(f'Total         : {n_normal + n_abnormal}')
print(f'Class ratio   : {n_normal}/{n_abnormal} = {n_normal/n_abnormal:.1f}:1')

In [ ]:
# load one normal and one abnormal sample
norm_file  = os.path.join(NORMAL_DIR,   '00000000.wav')
abnorm_file = os.path.join(ABNORMAL_DIR, '00000000.wav')

audio_n, sr_n = librosa.load(norm_file,   sr=16000, mono=True)
audio_a, sr_a = librosa.load(abnorm_file, sr=16000, mono=True)

print(f'Sample rate  : {sr_n} Hz')
print(f'Duration     : {len(audio_n)/sr_n:.1f} seconds')
print(f'Normal  - min:{audio_n.min():.3f}  max:{audio_n.max():.3f}  rms:{np.sqrt(np.mean(audio_n**2)):.4f}')
print(f'Abnormal- min:{audio_a.min():.3f}  max:{audio_a.max():.3f}  rms:{np.sqrt(np.mean(audio_a**2)):.4f}')

In [ ]:
# waveform plot — normal vs abnormal
t_audio = np.arange(len(audio_n)) / sr_n

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(t_audio, audio_n, linewidth=0.4, color='green')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Normal Machine Sound')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_audio, audio_a, linewidth=0.4, color='red')
axes[1].set_ylabel('Amplitude')
axes[1].set_title('Abnormal Machine Sound')
axes[1].set_xlabel('Time (s)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# log-mel spectrogram — normal vs abnormal
def compute_log_mel(audio, sr=16000):
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_fft=1024,
                                          hop_length=512, n_mels=64)
    return librosa.power_to_db(mel, ref=np.max)

mel_n = compute_log_mel(audio_n)
mel_a = compute_log_mel(audio_a)

print(f'Spectrogram shape: {mel_n.shape}  (mel_bands x time_frames)')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

img = librosa.display.specshow(mel_n, sr=16000, hop_length=512,
                                x_axis='time', y_axis='mel', ax=axes[0])
axes[0].set_title('Normal — Log-Mel Spectrogram')
fig.colorbar(img, ax=axes[0], format='%+2.0f dB')

img2 = librosa.display.specshow(mel_a, sr=16000, hop_length=512,
                                  x_axis='time', y_axis='mel', ax=axes[1])
axes[1].set_title('Abnormal — Log-Mel Spectrogram')
fig.colorbar(img2, ax=axes[1], format='%+2.0f dB')

plt.tight_layout()
plt.show()

In [ ]:
# class imbalance bar chart
plt.figure(figsize=(5, 4))
plt.bar(['Normal', 'Abnormal'], [n_normal, n_abnormal],
        color=['#4CAF50', '#F44336'], width=0.4)
plt.ylabel('Number of Files')
plt.title('MIMII Dataset — Class Distribution')
for i, v in enumerate([n_normal, n_abnormal]):
    plt.text(i, v + 3, str(v), ha='center', fontweight='bold')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# check a few more abnormal files to confirm they look different
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
axs = axes.flatten()

ab_files = sorted(os.listdir(ABNORMAL_DIR))[:6]
for ax, fname in zip(axs, ab_files):
    fpath = os.path.join(ABNORMAL_DIR, fname)
    audio, _ = librosa.load(fpath, sr=16000, mono=True)
    mel = compute_log_mel(audio)
    librosa.display.specshow(mel, sr=16000, hop_length=512, ax=ax)
    ax.set_title(fname, fontsize=8)
    ax.set_xlabel('')
    ax.set_ylabel('')

fig.suptitle('Sample Abnormal Spectrograms')
plt.tight_layout()
plt.show()

---
## Part 3 — Check Pre-Processed NPZ (CWRU)

In [ ]:
# the repo already has a pre-processed npz — let's check what's inside
npz_path = '../CWRU/CWRU_48k_load_1_CNN_data.npz'

if os.path.exists(npz_path):
    d = np.load(npz_path, allow_pickle=True)
    print('Keys:', list(d.keys()))
    for k in d.keys():
        print(f'  {k}: shape={d[k].shape}, dtype={d[k].dtype}')
else:
    print('File not found — will be created when preprocessing runs')

In [ ]:
# also check the CSV feature file
csv_path = '../CWRU/feature_time_48k_2048_load_1.csv'

if os.path.exists(csv_path):
    df_feat = pd.read_csv(csv_path)
    print('Shape:', df_feat.shape)
    print('\nColumns:', list(df_feat.columns[:10]), '...')
    print('\nSample rows:')
    print(df_feat.head())
else:
    print('CSV not found')

---
## Summary

| Dataset | Files | Sampling Rate | Format | Classes |
|---------|-------|---------------|--------|----------|
| CWRU    | 10 .mat files | 48,000 Hz | Vibration signal | 1 Normal + 9 Fault types |
| MIMII   | 381 normal + 138 abnormal .wav | 16,000 Hz | Machine audio | Binary (normal / abnormal) |

